**构建卷积神经网络**
* 卷积网络中的输入和层与传统神经网络有些区别，需重新设计， 训练模块基本一致

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets,transforms
import numpy as np
import matplotlib.pyplot as plt

d:\Anconda\envs\Paddle_tutorial\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**读取数据**
* 分别构建训练集和测试集（验证集）
* DataLoader来迭代这个数据

In [2]:
batch_size = 64 # 一个批次的大小
# 训练集
train_dataset = datasets.MNIST( root = './dataset',
                               train = True,
                               transform= transforms.ToTensor(),
                               download= True
)

# 测试集
test_dataset = datasets.MNIST( root = './dataset',
                               train = False,
                               transform= transforms.ToTensor(),
)

train_loader = torch.utils.data.DataLoader(dataset = train_dataset,
                                           batch_size = batch_size,
                                           shuffle = True
                                           )

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                          batch_size = batch_size,
                                          shuffle = True
)


**卷积网络模块构建**
* 一般卷积层， relu层， 池化层可以写成一个套餐
* 注意卷积最后结果还是一个特征图，需要把特征图转化成一个特征向量后才能做分类或者回归任务

In [3]:
class CNN(nn.Module):
    def __init__(self) -> None:
        super(CNN,self).__init__()

        self.conv1 = nn.Sequential( #输入大小 (1, 28, 28)
            nn.Conv2d(
                in_channels=1, #灰度图
                out_channels=16, #要得到多少个特征图
                kernel_size=5, #卷积核大小
                stride= 1, #步长
                padding=2, #如果希望卷积后大小要和原来的相同，需要设置padding=(kernel_size-1)/2 if stride=1
            ),   #输出特征图为(16, 28, 28)
            nn.ReLU(), #relu层
            nn.MaxPool2d(kernel_size=2) #进行池化操作（2x2区域），输出结果为：(16, 14, 14)
        
        )

        self.conv2 = nn.Sequential( #下一个套餐的输入(16,14,14)
            nn.Conv2d(16,32,5,1,2),  #输出 (32, 14, 14)
            nn.ReLU(),               #relu层
            nn.MaxPool2d(2)         # 输出 (32, 7, 7)
        )
        
        self.out = nn.Linear(32 * 7 * 7, 10) #全连接层得到的结果

    def forward(self, x):
        
        x = self.conv1(x)
        
        x = self.conv2(x)
        
        x = x.view(x.size(0), -1) # flatten操作， 结果为: (batch_size, 32 * 7 * 7)
        
        output = self.out(x) 
       
        return output


In [4]:
nn.Linear(16*14*14, 10)

Linear(in_features=3136, out_features=10, bias=True)

**准确率作为评估标准**

In [5]:
def accuracy(predictions, labels):
    pred = torch.max(predictions.data, 1)[1] 
    rights = pred.eq(labels.data.view_as(pred)).sum()
    return rights, len(labels)

**训练网络模型**

In [6]:
input_size = 28 #图像总尺寸28*28
num_classes = 10 #标签的种类数
num_epochs = 3 # 训练的总循环周期
batch_size = 64 # 一个批次的大小


# 实例化
net = CNN()
# 损失函数
criterion = nn.CrossEntropyLoss()
# 优化器
optimizer = optim.Adam(net.parameters(), lr = 0.001)

#开始循环训练
for epoch in range(num_epochs):
    # 当前epoch的结果保存下来
    train_rights = []

    for batch_idx, (data, target) in enumerate(train_loader): #针对容器中的每一个批进行循环
        net.train()
        output = net(data)
        loss = criterion(output, target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        right = accuracy(output, target)
        train_rights.append(right)

        if batch_idx % 100 == 0:

            net.eval()
            val_rights = []

            for (data, target) in test_loader:
                output = net(data)
                right = accuracy(output, target)
                val_rights.append(right)

            #准确率计算
            train_r = (sum([tup[0] for tup in train_rights]), sum ([tup[1] for tup in train_rights]))
            val_r = (sum([tup[0] for tup in val_rights]), sum ([tup[1] for tup in val_rights]))

            print("当前epoch:{} [{}/{} ({:.0f}%)]\t损失: {:.6f}\t训练集准确率: {:.2f}%\t测试集准确率: {:.2f}%".format(
                epoch, batch_idx*batch_size,len(train_loader.dataset),
                100 * batch_idx/len(train_loader),
                loss.data,
                100 * train_r[0].numpy() / train_r[1],
                100 * val_r[0].numpy() / val_r[1]
            ))


当前epoch:0 [0/60000 (0%)]	损失: 2.291248	训练集准确率: 15.62%	测试集准确率: 9.58%
当前epoch:0 [6400/60000 (11%)]	损失: 0.268726	训练集准确率: 76.02%	测试集准确率: 91.87%
当前epoch:0 [12800/60000 (21%)]	损失: 0.218077	训练集准确率: 84.24%	测试集准确率: 94.57%
当前epoch:0 [19200/60000 (32%)]	损失: 0.127709	训练集准确率: 87.86%	测试集准确率: 96.12%
当前epoch:0 [25600/60000 (43%)]	损失: 0.230814	训练集准确率: 89.83%	测试集准确率: 96.87%
当前epoch:0 [32000/60000 (53%)]	损失: 0.119665	训练集准确率: 91.17%	测试集准确率: 97.21%
当前epoch:0 [38400/60000 (64%)]	损失: 0.325811	训练集准确率: 92.12%	测试集准确率: 97.69%
当前epoch:0 [44800/60000 (75%)]	损失: 0.062383	训练集准确率: 92.85%	测试集准确率: 97.71%
当前epoch:0 [51200/60000 (85%)]	损失: 0.091373	训练集准确率: 93.41%	测试集准确率: 97.95%
当前epoch:0 [57600/60000 (96%)]	损失: 0.049504	训练集准确率: 93.87%	测试集准确率: 97.84%
当前epoch:1 [0/60000 (0%)]	损失: 0.074816	训练集准确率: 96.88%	测试集准确率: 98.03%
当前epoch:1 [6400/60000 (11%)]	损失: 0.029643	训练集准确率: 98.11%	测试集准确率: 97.95%
当前epoch:1 [12800/60000 (21%)]	损失: 0.010712	训练集准确率: 98.23%	测试集准确率: 98.23%
当前epoch:1 [19200/60000 (32%)]	损失: 0.027791	训练集准确率: 98.12%	测试集准确率